# ASISM v2 (learned ranker) على RunPod — شرح كامل خطوة بخطوة

**الحالة يوم 2026-10-04: مفيش أي خطوة GPU متوافق عليها. النوتبوك ده للقراءة والمراجعة دلوقتي.**

النوتبوك ده نسخة مشروحة من `docs/ham10000_asism_v2_learned_pod_commands.md`. الملف ده هو المرجع؛ لو اختلفوا، الملف هو الصح.

كل خلية كود هنا مقفولة بثلاث مفاتيح في أول خلية، وكلهم `False`:

| المفتاح | بيفتح إيه | مين يفتحه وإمتى |
|---|---|---|
| `READY` | أي أمر على الـ pod (حتى الفحص) | إنتِ، لما الـ pod يبقى شغّال والملفات مرفوعة |
| `APPROVED_MEASURE` | بدء القياس (GPU، حوالي 4 ساعات) | إنتِ، بعد ما موافقتك تتسجل في العقد §8 |
| `APPROVED_STAGE4` | بدء Stage 4 (GPU، حوالي 13 ساعة) | إنتِ، بعد ما تقري نتيجة الاختيار |

حتى لو فتحتِ `APPROVED_MEASURE` هنا، الكود نفسه بيرفض القياس طول ما `MEASUREMENT_APPROVED = False` في `scripts/followup/ham10000_asism_v2_utility.py`. يعني فيه قفلين مش قفل واحد.

## 1. الفكرة كلها في صفحة

عندنا 3,168 صورة صناعية آمنة (بعد استبعاد الصور البايظة والمكررة). السؤال: **أنهي صور نضيفها للتدريب، وكام صورة؟** ASISM v2 بيجاوب على السؤالين من غير ما حد يحدد رقم K أو نسبة.

الطريق ستّ مراحل:

1. **plan**: بنقسّم الصور الصناعية لثلاث أدوار (train 60%، validation 20%، test 20%) وبنرسم 200 مجموعة صغيرة منها بأحجام مختلفة (125 لحد 1,000 صورة).
2. **measure**: لكل مجموعة بندرّب classifier صغير وسريع (224 px، 300 خطوة) على «الصور الحقيقية + المجموعة»، 5 مرات بـ seeds مختلفة، ونقيس macro AUROC على `asism_tuning_heldout`. ده 200 × 5 = **1,000 تدريب**. الرقم ده هو «المنفعة» (utility) بتاعة المجموعة.
3. **G1**: فحص إن القياس نفسه ثابت. لو متوسط الـ 5 مرات مش متكرر كفاية (reliability أقل من 0.80)، الأرقام ضوضاء ومش هنتعلم منها، والطريق بيقف هنا والنتيجة دي بتتكتب زي ما هي.
4. **accept + fit**: بندرّب الـ ranker، وهو معادلة بتتوقع منفعة المجموعة من الإشارات الأربعة لصورها: `U(S) = b + λ·log(1 + Σ wᵢ)`. قبل ما نستخدمه بنختبره مرة واحدة على مجموعات الـ test اللي ما شافهاش. لو فشل، مش بيتستخدم، ودي النتيجة.
5. **select + stability**: 200 نسخة bootstrap من الـ ranker بتدّي لكل صورة «حد أدنى 95%» لفايدتها. كل class بيعرض صوره بالترتيب، وبنقف عند أول صورة حدّها الأدنى مش فوق الصفر. اللي اتاخد هو **C**، وعدده طالع من القاعدة مش من عندنا. بعدها بنعيد الاختيار بخمس fit seeds عشان نشوف العدد ثابت ولا لأ.
6. **Stage 4**: بندرّب الـ classifier الحقيقي (512 px، 3,000 خطوة، 20 seed) على أربع حالات:

| الحالة | بيتدرب على | ليه موجودة |
|---|---|---|
| A | الحقيقي بس | خط الأساس |
| B | الحقيقي + كل الـ 3,168 | هل «كله» أحسن من الاختيار؟ |
| C | الحقيقي + اختيار ASISM | الطريقة بتاعتنا |
| D | الحقيقي + عشوائي بنفس عدد C لكل class | بيثبّت العدد ويختبر «أنهي صور» بس |

المقارنة التأكيدية الوحيدة هي **C ضد D** على balanced accuracy. الباقي استكشافي.

## 2. كلمات هتقابلك

- **pod / volume**: الـ pod هو الجهاز اللي بتأجّريه بالساعة. الـ volume هو القرص اللي بيفضل بعد ما الـ pod يتقفل، ومتركّب على `/workspace`.
- **bundle**: ملف واحد فيه الكود وتاريخه. بنستخدمه لأن الـ repo خاص والـ SSH مقفول على شبكتك.
- **run line**: فرع اسمه `run/...` فيه بالظبط الكود اللي هيشتغل على الـ pod.
- **frozen / written once**: ملف بيتكتب مرة واحدة والكود بيرفض يكتبه تاني. ده مقصود: النتيجة ما تتغيرش بعد ما تتقري.
- **resume**: لو الـ pod وقع في نص القياس، نفس الأمر بيكمّل من أول خلية ناقصة.
- **MONITORING**: مقارنة على `classifier_val`. مش التقييم النهائي ومش بتقرر حاجة.

## 3. خلية المفاتيح (شغّليها أول حاجة في كل جلسة)

`sh()` بتشغّل أمر على الـ pod بعد ما تحمّل البيئة والمتغيرات الثلاثة (`NS`، `CAND`، `SCORES`) بنفسها، فمفيش أمر هيشتغل ومتغير فاضي.

In [ ]:
import subprocess

READY = False             # الـ pod شغّال والملفات مرفوعة
APPROVED_MEASURE = False  # موافقتك على القياس متسجلة في العقد §8
APPROVED_STAGE4 = False   # قريتِ نتيجة الاختيار ووافقتِ على Stage 4

ENV = ("source /workspace/env.sh && cd $PROJECT_ROOT && export NS=ham-stratified-v1 "
       "CAND=$PROJECT_ROOT/outputs/ham10000/stage2/ham-stratified-v1/all_candidates.csv "
       "SCORES=/workspace/signals/four_signal/gonogo_signal_set && ")
RANKER = "$PROJECT_ROOT/outputs/ham10000/stage3_asism_v2_ranker/$NS"
STAGE4 = "$PROJECT_ROOT/outputs/ham10000/stage4_asism_v2_learned/$NS"
SWITCHES = {None: lambda: True, "measure": lambda: APPROVED_MEASURE, "stage4": lambda: APPROVED_STAGE4}


def sh(command, needs=None):
    """Run one command on the pod. Returns the exit code; prints what the command printed."""
    if not READY:
        raise RuntimeError("READY = False: مفيش أمر بيشتغل. افتحيه لما الـ pod يبقى جاهز.")
    if not SWITCHES[needs]():
        raise RuntimeError(f"الخطوة دي ({needs}) محتاجة موافقتك ومفتاحها لسه False.")
    done = subprocess.run(["bash", "-c", ENV + command], text=True, capture_output=True)
    print(done.stdout + done.stderr)
    return done.returncode


def running(pattern):
    """True if a process whose command line matches is alive. [x]yz keeps the check from matching itself."""
    probe = subprocess.run(["bash", "-c", f"pgrep -af '[{pattern[0]}]{pattern[1:]}'"], text=True, capture_output=True)
    print(probe.stdout or "مفيش حاجة شغّالة")
    return bool(probe.stdout.strip())

## 4. الخطوة 0 — على اللابتوب، قبل ما تفتحي pod (مفيش فلوس)

الأوامر دي **مش بتتشغل من النوتبوك**؛ دي على اللابتوب من فولدر المشروع، وبيئة `asism` مفعّلة.

**الترتيب مهم.** الـ pod بيشغّل الكود اللي بيوصله، والقياس بيقرا `MEASUREMENT_APPROVED` من الكود ده. فلازم:

1. موافقتك تتكتب في العقد §8 بتاريخها (commit لوحده).
2. `MEASUREMENT_APPROVED = True` (commit تاني لوحده).
3. الاتنين على فرع `run/ham10000-asism-v2-learned`.
4. بعدها يتعمل الـ bundle. لو اتعمل قبل كده، القياس هيترفض على الـ pod.

الاختبارات الكاملة (وقّفي مزامنة OneDrive الأول):

```bash
python tests/run_all.py
```

الـ bundle، وبعده رقم الـ commit اللي هتقارني بيه على الـ pod:

```bash
git bundle create C:/Users/walaa/ham10000_work/bundles/ham10000-asism-v2-learned.bundle run/ham10000-asism-v2-learned
```

```bash
git rev-parse --short run/ham10000-asism-v2-learned
```

**اللي بيترفع** من متصفح ملفات Jupyter على الـ pod، في `/workspace/upload/`:

- `ham10000-asism-v2-learned.bundle`
- الثمان ملفات اللي في `C:\Users\walaa\ham10000_work\v3_signals\four_signal\gonogo_signal_set\`

**الـ pod نفسه:** لازم يتفتح على نفس الـ network volume اللي عليه `/workspace/master`، لأن ملف الـ candidates المجمّد مكتوب فيه مسارات كاملة تحت `/workspace/master/...`. والتوقيتات محسوبة على RTX 5090.

## 5. الخطوة 1 — فحص الـ pod (CPU، مفيش تدريب)

الهدف: نتأكد إن كل حاجة القياس هيقراها موجودة وهي هي. **لو أي رقم طلع غير المتوقع، نقف.**

أول خلية: حالة الكود الموجود على الـ pod قبل ما نغيّره. المتوقع إن `git status` ما يطبعش تعديلات.

In [ ]:
sh("git status --short && git log --oneline -1")

نجيب الكود الجديد من الـ bundle. المتوقع: آخر سطر هو نفس الـ commit اللي كتبتيه في الخطوة 0.

In [ ]:
sh("git fetch /workspace/upload/ham10000-asism-v2-learned.bundle run/ham10000-asism-v2-learned:run/ham10000-asism-v2-learned && git checkout run/ham10000-asism-v2-learned && git log --oneline -1")

ملفات الإشارات تروح مكانها. المتوقع بعد `ls`: ثمان ملفات.

In [ ]:
sh("mkdir -p $SCORES && cp /workspace/upload/*_scores.parquet /workspace/upload/*_scores.provenance.json $SCORES/ && ls $SCORES")

بصمة ملف الـ candidates. لازم تطلع بالظبط:

`bf8047b5dde96259c1268e5d8a3db6bac0d37392088df45182fde350ac7ef51a`

الكود بيرفض أي ملف تاني، فالفحص ده بيوفّر عليكِ خطأ بعدين.

In [ ]:
sh("sha256sum $CAND")

الصور الصناعية كلها موجودة؟ المتوقع: `3168 candidates, 0 missing`.

In [ ]:
sh("python -c \"import pandas as pd, pathlib as p; d=pd.read_csv('$CAND'); print(len(d), 'candidates,', sum(not p.Path(x).is_file() for x in d.image_path), 'missing')\"")

صور الـ splits. المتوقع على الأقل: `classifier_train` 1,641، `classifier_val` 1,401، `asism_tuning_heldout` 1,377.

In [ ]:
sh("for s in classifier_train classifier_val asism_tuning_heldout; do echo $s $(ls $PROJECT_ROOT/data/ham10000/processed/images/$NS/$s | wc -l); done")

الـ GPU ظاهر؟ المتوقع: `True` واسم الكارت. لو `False` نقف: التدريب هيشتغل على الـ CPU من غير ما يشتكي.

In [ ]:
sh("python -c \"import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))\"")

الاختبارات الكاملة على الـ pod (دقايق). المتوقع: `0 failed`.

In [ ]:
sh("python tests/run_all.py 2>&1 | tail -5")

## 6. الخطوة 2 — plan (CPU، ثواني)

بيجمّد الأدوار، الـ 200 مجموعة، والـ 1,000 خلية (مجموعة × seed) في `utility_plan.json`. **بيتكتب مرة واحدة**؛ تشغيله تاني بيترفض، وده مقصود عشان المجموعات ما تترسمش تاني بعد ما نشوف نتايج.

المتوقع في الناتج: `runs_planned: 1000`.

In [ ]:
sh("python -m scripts.followup.ham10000_asism_v2_utility --phase plan --candidates $CAND --scores-dir $SCORES")

## 7. الخطوة 3 — measure (GPU، حوالي 4 ساعات، حوالي 3 دولار) — محتاجة موافقتك

**ممنوع يتشغّل مرتين في نفس الوقت.** نسختين هيكتبوا نفس الخلايا، و G1 بيرفض أي خلية متكررة، وملفات القياس دليل وما بتتعدّلش بالإيد. عشان كده خلية البدء بتفحص الأول وبترفض لو فيه نسخة شغّالة.

نفس الخلية هي **الاستئناف**: لو الـ pod وقع، افتحيه تاني وشغّلي الخلية دي؛ هتكمّل من أول خلية ناقصة.

In [ ]:
if running("ham10000_asism_v2_utility"):
    raise RuntimeError("القياس شغّال بالفعل. ما تبدأيش نسخة تانية.")
sh("setsid nohup python -m scripts.followup.ham10000_asism_v2_utility --phase measure --candidates $CAND "
   "--i-understand-this-trains-real-models < /dev/null >> /workspace/asism_v2_measure.log 2>&1 &", needs="measure")

**المتابعة** (شغّليها بعد دقيقتين، وبعدين كل شوية):

- أول سطرين في الـ log: `measure inputs: frozen` (أو `identical` في الاستئناف) و `1000 of 1000 utility runs pending on <اسم الكارت>`.
- **لو مكتوب `on cpu`:** وقّفي فورًا بالخلية اللي بعدها.
- كل سطر بعد كده حوالي 14 ثانية وفيه `macro_auroc_ovr=...`.
- لو ظهر `has NOT been approved`: الـ bundle اتعمل قبل commit الموافقة. مفيش حاجة اتدربت.

In [ ]:
sh("tail -n 12 /workspace/asism_v2_measure.log; wc -l " + RANKER + "/utility_runs_*.jsonl")

إيقاف طارئ فقط (مثلًا لو شغّال على CPU). الخلايا اللي خلصت بتفضل محفوظة.

In [ ]:
# sh("pkill -f ham10000_asism_v2_utility")   # شيلي علامة # لو محتاجة توقّفي

**خلص إمتى؟** لما الملفين مع بعض يبقوا 1,000 سطر (800 fit + 200 test) والخلية دي تطبع «مفيش حاجة شغّالة».

**لو الاستئناف رفض:**

| الرسالة | معناها | تعملي إيه |
|---|---|---|
| `measure inputs changed since the first run` | حاجة اتغيرت عن أول تشغيل (split، candidates، plan، أو كود القياس) | قفي واعرفي إيه اللي اتغير. ما تمسحيش `measure_inputs.json` |
| خطأ JSON وهو بيقرا ملف القياس | الـ pod وقع وهو بيكتب سطر | قفي وبلّغي. الملف ما بيتصلّحش بالإيد |

In [ ]:
running("ham10000_asism_v2_utility")

## 8. الخطوة 4 — G1 (CPU، ثواني)

بيحسب ثبات القياس على مجموعات الـ train والـ validation بس (الـ test لسه ما اتفتحش). بيتكتب مرة واحدة في `g1_reliability.json`.

- `passed: true` → نكمّل.
- `passed: false` → **الطريق خلص هنا، ودي النتيجة.** مش بنغيّر الحد (0.80) ومش بنعيد القياس.

In [ ]:
sh("python -m scripts.followup.ham10000_asism_v2_utility --phase gate")

## 9. الخطوة 5 — accept ثم fit ثم select ثم stability (CPU، حوالي ساعة ونص)

كل مرحلة بترفض تشتغل لو اللي قبلها ما نجحتش.

**accept**: بيقرا مجموعات الـ test **مرة واحدة**. الـ ranker لازم يرتّب المجموعات صح جوه كل حجم (Spearman ≥ 0.50، p ≤ 0.05) ويغلب النموذج اللي من غير إشارات. لو `accepted: false`: الترتيب المتعلَّم ما بيتستخدمش، مفيش حاجة تحت بتتشغل، ودي النتيجة.

In [ ]:
sh("python -m scripts.followup.ham10000_asism_v2_learned_select --phase accept --candidates $CAND --scores-dir $SCORES")

**fit**: 200 نسخة bootstrap من الـ ranker. دي أطول خلية.

In [ ]:
sh("python -m scripts.followup.ham10000_asism_v2_learned_select --phase fit --candidates $CAND --scores-dir $SCORES")

**select**: بيطبّق قاعدة الوقوف وبيكتب `c_selected.csv` وملف D لكل seed (نفس عدد C لكل class).

In [ ]:
sh("python -m scripts.followup.ham10000_asism_v2_learned_select --phase select --candidates $CAND --scores-dir $SCORES")

**stability**: نفس الاختيار بخمس fit seeds. بيقول العدد بيتحرك قد إيه. تقرير بس، مش شرط.

In [ ]:
sh("python -m scripts.followup.ham10000_asism_v2_learned_select --phase stability --candidates $CAND --scores-dir $SCORES")

## 10. الخطوة 6 — قفي واقري الاختيار

`selection_outcome` هو اللي بيحدد بروتوكول Stage 4. محدش بيختاره:

| `selection_outcome` | البروتوكول | اللي بيتدرب |
|---|---|---|
| `subset` | `asism_v2_learned` | A, B, C, D |
| `all` أو `none` | `asism_v2_learned_all_or_none` | A, B بس (C هي B أو A بالتعريف) |

نزّلي فولدر `outputs/ham10000/stage3_asism_v2_ranker/ham-stratified-v1/` كله قبل ما تكمّلي. **ممكن تقفلي الـ pod هنا.** Stage 4 محتاج موافقة لوحده.

In [ ]:
sh("python -c \"import json; m=json.load(open('" + RANKER + "/asism_v2_learned_selection_manifest.json')); print(m['selection_outcome'], m['n_selected_c'], m['per_class'])\"")

## 11. الخطوة 7 — Stage 4 (GPU، حوالي 13 ساعة، حوالي 9 دولار) — موافقة لوحدها

اختاري السطر اللي يطابق `selection_outcome`. التدريب اللي كتب `run_manifest.json` بيتخطّى، فالخلية آمنة للاستئناف.

In [ ]:
PROTO, CONDS = "asism_v2_learned", "A B C D"                    # selection_outcome = subset
# PROTO, CONDS = "asism_v2_learned_all_or_none", "A B"          # selection_outcome = all أو none

if running("ham10000_train_conditions"):
    raise RuntimeError("Stage 4 شغّال بالفعل.")
loop = ("for c in " + CONDS + "; do for s in $(seq 42 61); do [ -f " + STAGE4 + "/$c/seed$s/run_manifest.json ] || "
        "python scripts/classify/ham10000_train_conditions.py --protocol " + PROTO + " --condition $c --seed $s || exit 1; done; done")
sh("setsid nohup bash -c '" + loop + "' < /dev/null >> /workspace/asism_v2_stage4.log 2>&1 &", needs="stage4")

المتابعة: عدد التدريبات اللي خلصت. المتوقع في الآخر 80 (أو 40 لو A و B بس).

In [ ]:
sh("ls " + STAGE4 + "/*/seed*/run_manifest.json | wc -l; tail -n 3 /workspace/asism_v2_stage4.log")

## 12. الخطوة 8 — التجميع والمقارنة على classifier_val (CPU)

المقارنة دي **MONITORING**: على split المتابعة، مش التقييم النهائي، ومش بتقرر حاجة. فيها المقاييس الأربعة القديمة والثلاثة الجديدة (Macro AP، Brier، ECE). في Brier و ECE الأقل أحسن.

التقييم النهائي على `final_eval_heldout` (Stage 5) **مش هنا**؛ مستني قرار المشرف في سياسة الـ test set.

In [ ]:
sh("python scripts/classify/ham10000_aggregate_conditions.py --protocol " + PROTO)

In [ ]:
sh("python -m scripts.followup.ham10000_asism_v2_compare --protocol " + PROTO + " --source classifier_val")

## 13. الخطوة 9 — التنزيل وقفل الـ pod

نزّلي:

- `outputs/ham10000/stage3_asism_v2_ranker/ham-stratified-v1/` كله
- `outputs/ham10000/stage4_asism_v2_learned/ham-stratified-v1/` من غير ملفات `model.pt`
- `/workspace/asism_v2_measure.log` و `/workspace/asism_v2_stage4.log`

وبعدها اقفلي الـ pod من لوحة RunPod. الـ pod بيتحاسب طول ما هو شغّال، حتى لو مفيش تدريب.

## 14. ملخّص: إيه اللي ممكن يوقّف الطريق، وكله نتيجة مش خطأ

| فين | الشرط | لو ما اتحققش |
|---|---|---|
| G1 | reliability ≥ 0.80 | مش بنتعلم من القياس. دي النتيجة |
| accept | Spearman ≥ 0.50، p ≤ 0.05، أحسن من «الحجم والـ class بس» | الترتيب المتعلَّم ما بيتستخدمش. دي النتيجة |
| select | النتيجة `all` أو `none` | مفيش C و D؛ بيتدرب A و B بس |

ولا حد من الحدود دي بيتغير بعد ما الأرقام تتقري. أي تغيير هو تعديل مكتوب بتاريخه قبل التشغيل اللي بعده.